# Download BRIGHT to Google Drive

This notebook stores the BRIGHT archives and extracted data in Google Drive. It does not require GitHub authentication.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess
import hashlib
import zipfile

BRIGHT_DIR = Path("/content/drive/MyDrive/disaster-lens/data/raw/bright")
ARCHIVE_DIR = BRIGHT_DIR / "_archives"
BRIGHT_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

files = {
    'post-event.zip': '13dbfff273e95995fee2a868388da4ea',
    'target.zip': 'd7f48f686e0b01772949c1b5e56e3146',
    'pre-event.zip': '087db04490233e40fd5b53ea1d3b374a',
}
base_url = "https://huggingface.co/datasets/Kullervo/BRIGHT/resolve/main"

def file_md5(path: Path) -> str:
    digest = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

for filename, expected_md5 in files.items():
    archive_path = ARCHIVE_DIR / filename
    url = f"{base_url}/{filename}?download=true"
    print(f"Downloading {filename}...")
    subprocess.run(["wget", "-c", "--show-progress", url, "-O", str(archive_path)], check=True)
    actual_md5 = file_md5(archive_path)
    if actual_md5 != expected_md5:
        raise RuntimeError(f'{filename} checksum mismatch: {actual_md5} != {expected_md5}')
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(BRIGHT_DIR)

print(f"BRIGHT data is available at: {BRIGHT_DIR}")
print(f"Set DISASTERLENS_BRIGHT_ROOT to: {BRIGHT_DIR}")

In [ ]:
import os

os.environ["DISASTERLENS_BRIGHT_ROOT"] = "/content/drive/MyDrive/disaster-lens/data/raw/bright"
print(os.environ["DISASTERLENS_BRIGHT_ROOT"])